# Compliance/HR Agent Demo

The Compliance/HR Agent is the seventh and final specialist — the roster is now complete.
Registered alongside Finance, Sentiment, Sales, Operations, Growth, and Risk in the boss
agent's roster (`backend/agents/boss/registry.py`).

This agent is architecturally different from the other six: instead of querying Delta tables
for computed metrics, most of its tools do **precision fact retrieval** against three
fictional institutional documents (per CLAUDE.md section 2, a wrapper entity "Olist Inc." -
not real company data), built for this agent specifically:

- `company_registration.md` — incorporation facts, jurisdiction, status
- `hr_policy.md` — employment terms, leave, code of conduct, compensation, termination
- `vendor_contract.md` — the standard seller agreement template, including an SLA clause

## Its 7 tools

| Tool | Computes | Data reality |
|---|---|---|
| `search_policy_docs` | Keyword search across all 3 documents | Proxy for real semantic search — Vector Search with `doc_type` metadata tagging (a separate collection from reviews per CLAUDE.md) is build-order step 6, not built yet |
| `get_company_registration_info` | All registration fields | Direct structured lookup — this is the "precision fact-retrieval, not pattern retrieval" CLAUDE.md calls for with these docs |
| `check_contract_clause` | Looks up a clause by topic for a given seller | All sellers share ONE standard template — no bespoke per-vendor contracts exist. The `vendor` param validates the seller_id and is accepted for interface consistency with a future state where addenda might exist |
| `check_policy_compliance` | Looks up whether an HR topic is documented | Direct structured lookup |
| `contract_expiry_tracker` | Sellers approaching their annual contract renewal | Renewal date computed from REAL seller onboarding dates (first order date) + the contract's stated annual cadence |
| `policy_gap_analysis` | Which expected HR topics are missing | **Deliberately finds real gaps** — see below |
| `cross_reference_sla_compliance` | Seller's actual on-time % vs. the contractual SLA threshold | **The flagship cross-agent tool** — see below |

## A deliberately incomplete HR policy, so `policy_gap_analysis` has something real to find

The HR policy document covers 5 of 9 standard topics. Four are missing on purpose:
**Data Privacy Policy, Anti-Discrimination & Equal Opportunity, Remote Work Policy, Grievance
Procedure**. If every topic were present, `policy_gap_analysis()` would always return "no
gaps" — a demo that can't fail isn't demonstrating anything. Note the tool's own scope: it
only reports documentation coverage, it does not recommend or make employment decisions —
CLAUDE.md section 6 explicitly scopes this agent away from that (EU AI Act high-risk category).

## The flagship cross-agent tool

CLAUDE.md calls out `cross_reference_sla_compliance()` specifically: *"No single agent can
answer this alone — build this carefully as it's the strongest demo of why the multi-agent
design earns its complexity."* It:

1. Parses the SLA threshold **directly out of the contract text** via regex (`no less than
   85%`) rather than hardcoding a duplicate number — if the contract document changes, the
   check stays in sync automatically.
2. Computes a specific seller's **actual** on-time delivery rate directly against the Delta
   tables, using the same on-time definition as the Operations Agent's
   `calculate_delivery_delay`/`seller_performance_score` (kept self-contained per this
   codebase's convention — no cross-package Python import — but the same underlying
   methodology).
3. Compares the two and returns a compliant/breach verdict a single agent's tools could never
   produce alone: Operations doesn't know the contractual threshold, and Compliance doesn't
   compute delivery metrics anywhere else.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from backend.agents.compliance import ComplianceAgent
from backend.db import DatabricksClient

db = DatabricksClient()
agent = ComplianceAgent(db)

## Run the agent against live Delta tables + local documents

In [2]:
briefing = agent.run("Are we compliant with our seller agreements and HR policy?")

print(f"Agent: {briefing.agent.value}")
print(f"Execution time: {briefing.execution_time_ms:.0f}ms")
print(f"Tool calls: {len(briefing.tool_calls)} ({sum(1 for t in briefing.tool_calls if t.success)} succeeded)")

Agent: Compliance
Execution time: 5948ms
Tool calls: 7 (7 succeeded)


## Inspect the findings

In [3]:
for f in briefing.findings:
    print(f"[{f.severity.upper()}] (confidence {f.confidence:.2f}) {f.claim}")
    print(f"  source: {f.source}\n")

[INFO] (confidence 0.40) Keyword search for 'Are we compliant with our seller agreements and HR policy?' matched 0 document sections
  source: search_policy_docs

[INFO] (confidence 1.00) Company registration status: Active, in good standing
  source: get_company_registration_info

[WARNING] (confidence 0.90) HR policy covers 55.6% of expected topics; missing: Data Privacy Policy, Anti-Discrimination & Equal Opportunity, Remote Work Policy, Grievance Procedure
  source: policy_gap_analysis

[INFO] (confidence 0.85) Seller contract SLA clause found: Service Level Agreement (SLA)
  source: check_contract_clause

[INFO] (confidence 0.90) HR policy on 'Leave Policy' is documented
  source: check_policy_compliance

[WARNING] (confidence 0.75) 154 of 3053 seller agreements renew within 30 days
  source: contract_expiry_tracker

[INFO] (confidence 0.85) Seller 6560211a19b47992c3666cc44a7e94c0 is compliant with the 85.0% SLA at 93.79% actual on-time delivery
  source: cross_reference_sla_compl

## Inspect the SLA cross-reference directly

In [4]:
from backend.agents.compliance import tools as compliance_tools

# Highest-volume seller, same pick the agent's own analyze() call makes
top_seller = db.query(f"SELECT seller_id FROM {db.table('order_items')} GROUP BY seller_id ORDER BY COUNT(*) DESC LIMIT 1")[0]["seller_id"]
sla = compliance_tools.cross_reference_sla_compliance(db, vendor=top_seller)
print(sla)

{'vendor': '6560211a19b47992c3666cc44a7e94c0', 'found': True, 'actual_on_time_pct': 93.79, 'contractual_threshold_pct': 85.0, 'compliant': True, 'delivered_order_count': 1819, 'sla_clause_source': 'vendor_contract.md - Service Level Agreement (SLA)', 'note': 'threshold parsed directly from the contract text, not hardcoded - stays in sync if the contract document changes'}


## Through the boss agent — all 7 specialists now available

The roster is complete. A query that spans compliance, operations, and finance is the real
test of whether the multi-agent design holds together at full scale.

In [5]:
from backend.agents.boss import BossAgent

boss = BossAgent(db)
rec = boss.run("Is our top seller compliant with their contract, and what's the financial exposure if they're not?")

print(f"Agents invoked: {[a.value for a in rec.agents_invoked]}")
print(f"Overall confidence: {rec.confidence_overall}")
print(f"\nSynthesis:\n{rec.synthesis}")

if rec.dissents:
    print("\nDissents:")
    for d in rec.dissents:
        print(f"  - {[a.value for a in d.agents_involved]}: {d.summary}")

Agents invoked: ['Compliance', 'Finance']
Overall confidence: 0.78

Synthesis:
To: Executive Board
From: Chairperson, AI Boardroom
Subject: Compliance status of top seller and associated financial exposure

1. **Seller compliance** – The Compliance Agent, using the `check_contract_clause` and `cross_reference_sla_compliance` tools, confirmed that Seller 6560211a19b47992c3666cc44a7e94c0 meets the Service Level Agreement requirement. The seller achieved a 93.79% on‑time delivery rate, exceeding the 85.0% threshold set in the contract. Therefore, the seller is currently compliant with the contractual SLA.

2. **Financial exposure** – Finance’s `refund_impact_analysis` tool reports that 1.68% of total payment value, amounting to $269,735.11, is at risk from canceled or unavailable orders. Additionally, the `detect_revenue_anomalies` tool identified 13 daily revenue anomalies in a 614‑day period, suggesting potential irregularities in transaction reporting. Because the `calculate_cogs` tool

## Roster complete

All 7 specialist agents are built and registered: Finance, Sentiment, Sales, Operations, Growth, Risk, Compliance/HR. Per CLAUDE.md's build order, what's left:

6. RAG layer (Databricks Vector Search) for Sentiment's review search and Compliance's document search — both currently keyword-match proxies
7. Governance logging middleware persistence + MongoDB
8. MERN frontend + streaming + WebSocket agent status
9. Eval suite
10. Deployment containerization